In [ ]:
import pandas as pd # import your library

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhe-CUNY/python-pandas-geopandas/blob/main/01_pandas.ipynb) 01_pandas.ipynb

In [ ]:
# set up your data full of columns and rows, pandas is most happy when your data types are the same for the column
data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva'],
    'Age': [24, 27, 22, 32, 29],
    'City': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix']
}

df = pd.DataFrame(data)
df

In [ ]:
# select a column
df['Name']

In [ ]:
# run a method on a column
df['Name_Upper'] = df['Name'].str.upper()

In [ ]:
df

In [ ]:
df['Age'].sum()

### Reading a CSV File
You can read a CSV file into a DataFrame using the pd.read_csv() function. It can even be from a URL

In [ ]:
# we can modify the url to prefilter our data, but for now let's pull the latest 10k records 
url = 'https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$limit=10000'
df = pd.read_csv(url)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# view only some columns

df[['agency_name','complaint_type','incident_zip']]

In [ ]:
# view only some rows

df[['agency_name','complaint_type','incident_zip']][100:102]

### Filters

In [ ]:
df['agency_name']

In [ ]:
df['agency_name'].value_counts()

In [ ]:
df[df['agency_name'] == 'New York City Police Department']

In [ ]:
# what complaint_types are there

df[df['agency_name'] == 'New York City Police Department']['complaint_type'].value_counts()

In [ ]:
# more than one filter use ()
# & <-- for and
# | <-- for or

df[(df['agency_name'] == 'New York City Police Department') & (df['complaint_type'] == 'Noise - Residential')]

In [ ]:
# get all the noise complaints

df[
    (df['agency_name'] == 'New York City Police Department') & 
    (
        (df['complaint_type'] == 'Noise - Residential') | 
        (df['complaint_type'] == 'Noise - Street/Sidewalk') |
        (df['complaint_type'] == 'Noise - Commercial') | 
        (df['complaint_type'] == 'Noise - Vehicle')
    )
    ]

In [ ]:
# other ways

df[
    (df['agency_name'] == 'New York City Police Department') & 
    (df['complaint_type'].isin(['Noise - Residential', 'Noise - Street/Sidewalk', 'Noise - Commercial', 'Noise - Vehicle']))
    ]

In [ ]:
# other ways

df2 = df[
    (df['agency_name'] == 'New York City Police Department') & 
    (df['complaint_type'].str.startswith('Noise'))
    ]

In [ ]:
df2

### Sorting

In [ ]:
# lets find the latest complaint

df2['created_date'] # note that it is type string so if we sort ... it will still work

In [ ]:
df2.sort_values?

In [ ]:
df2.sort_values(by = 'created_date')[:2] # get the first 2 records

In [ ]:
df2.sort_values(by = 'created_date', ascending = False)[:2] # sort the other way

In [ ]:
# convert the data types for a better filter and readable date time

df2['date_objects'] = pd.to_datetime(df['created_date'])
df2['date_objects']

In [ ]:
df2[['agency_name','complaint_type','date_objects']].sort_values(by = 'date_objects')[:2] 

### Group by 

In [ ]:
# let's try to answer when noise complaints are happening

# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html
df2[['unique_key','date_objects']].groupby('date_objects').count()

In [ ]:
# get time
df2['date_objects'].dt.time

In [ ]:
df2['date_objects'].dt.hour

In [ ]:
# create a new column to store it
df2['hour'] = df2['date_objects'].dt.hour

In [ ]:
df2[['unique_key','hour']].groupby('hour').count()

In [ ]:
# sort
df2[['unique_key','hour']].groupby('hour').count().sort_values(by = 'unique_key', ascending = False)

### Pivots

In [ ]:
pd.pivot_table?

In [ ]:
pd.pivot_table(
    df2,
    values='unique_key',
    index='hour',
    aggfunc='count'
).sort_values(by='unique_key',ascending = False  )

In [ ]:
pd.pivot_table(
    df2,
    values='unique_key',
    columns='complaint_type',
    index='hour',
    aggfunc='count'
).fillna(0) # we want to fill in all the NaN (points without data with 0s)